# Solving Activation Functions

## Generating Lookup Tables for the Output Functions

In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import torch
from torch import nn

### Below are the implementations of the five activation functions using LUTs

Sigmoid: 

In [ ]:
def sigmoid_generator(inwidth, outwidth, beta):
    my_sigmoid = {}

    inwidth = int(np.exp2(inwidth))
    outwidth = int(np.exp2(outwidth))
    beta = int(np.exp2(beta)) # this is 2^(int_scale+weight_scale) # 6 + 4 = 0

    for i in range(round(-inwidth/2), round(inwidth/2)):
        my_sigmoid[i] = beta/(1 + np.exp(-i/beta)) 

    
    plt.plot(list(my_sigmoid.values()))
    plt.title("Sigmoid")
    plt.xlabel("Input index")
    plt.ylabel("Sigmoid érték")
    plt.grid()
    plt.show()

    beta = int(np.log2(beta))
    with open(f"my_sgimoid_lookup_table_{beta}.txt", "w") as f:
        for key in sorted(my_sigmoid.keys()):
            f.write(f"{my_sigmoid[key]},\n")

    # shift it onto the positive half-axis
    offset = inwidth//2
    new_d = {k + offset: v for k, v in my_sigmoid.items()}
    my_sigmoid = new_d

    return my_sigmoid

Softplus:

In [ ]:
def softplus_generator(inwidth, outwidth, beta):
    my_softplus = {}

    inwidth = int(np.exp2(inwidth))
    outwidth = int(np.exp2(outwidth))
    beta = int(np.exp2(beta))

    function = nn.Softplus(1/beta)

    for i in range(round(-inwidth/2), round(inwidth/2)):
        x = torch.tensor(float(i))
        my_softplus[i] = function(x).item()

    plt.plot(list(my_softplus.values()))
    plt.title("Softplus")
    plt.xlabel("Input index")
    plt.ylabel("Softplus érték")
    plt.grid()
    plt.show()

    beta = int(np.log2(beta))

    with open(f"my_softplus_lookup_table_{beta}.txt", "w") as f:
        for key in sorted(my_softplus.keys()):
            f.write(f"{my_softplus[key]},\n")

    offset = inwidth//2
    new_d = {k + offset: v for k, v in my_softplus.items()}
    my_softplus = new_d

    return my_softplus

Tanh:

In [ ]:
def tanh_generator(inwidth, outwidth, beta):
    my_tanh = {}

    inwidth = int(np.exp2(inwidth))
    outwidth = int(np.exp2(outwidth))
    beta = int(np.exp2(beta))

    for i in range(round(-inwidth/2), round(inwidth/2)):
        my_tanh[i] = beta*np.tanh(i/beta)
    
    beta = np.log2(beta)

    plt.plot(list(my_tanh.values()))
    plt.title("Tanh")
    plt.xlabel("Bemenet")
    plt.ylabel("Tanh érték")
    plt.grid()
    plt.show()

    with open("my_tanh_lookup_table.txt", "w") as f:
        for key in sorted(my_tanh.keys()):
            f.write(f"{my_tanh[key]},\n")

    offset = inwidth//2
    new_d = {k + offset: v for k, v in my_tanh.items()}
    my_tanh = new_d

    return my_tanh

Integer LUT generators. These are needed so that, if possible, we store integers in the FPGA memory rather than fixed-point values.

In [ ]:
def softplus_generator_int(inwidth, outwidth, beta):
    my_softplus = {}

    inwidth = int(np.exp2(inwidth))
    outwidth = int(np.exp2(outwidth))
    beta = int(np.exp2(beta))

    function = nn.Softplus(1/beta)

    for i in range(round(-inwidth/2), round(inwidth/2)):
        x = torch.tensor(float(i))
        my_softplus[i] = function(x).item()

    plt.plot(list(my_softplus.values()))
    plt.title("Softplus")
    plt.xlabel("Bemenet")
    plt.ylabel("Softplus érték")
    plt.grid()
    plt.show()

    beta = int(np.log2(beta))

    with open(f"my_softplus_lookup_table_{beta}.txt", "w") as f:
        for key in sorted(my_softplus.keys()):
            f.write(f"{int(np.round(my_softplus[key]))},\n")

    offset = inwidth//2
    new_d = {k + offset: v for k, v in my_softplus.items()}
    my_softplus = new_d

    return my_softplus

In [ ]:
def sigmoid_generator_int(inwidth, outwidth, beta):
    my_sigmoid = {}

    inwidth = int(np.exp2(inwidth))
    outwidth = int(np.exp2(outwidth))
    beta = int(np.exp2(beta)) # this is 2^(int_scale+weight_scale) # 6 + 4 = 0

    for i in range(round(-inwidth/2), round(inwidth/2)):
        my_sigmoid[i] = beta/(1 + np.exp(-i/beta)) 

    plt.plot(list(my_sigmoid.values()))
    plt.title("Sigmoid")
    plt.xlabel("Bemenet")
    plt.ylabel("Sigmoid érték")
    plt.grid()
    plt.show()

    beta = int(np.log2(beta))
    with open(f"my_sgimoid_lookup_table_{beta}.txt", "w") as f:
        for key in sorted(my_sigmoid.keys()):
            f.write(f"{int(np.round(my_sigmoid[key]))},\n")

    offset = inwidth//2
    new_d = {k + offset: v for k, v in my_sigmoid.items()}
    my_sigmoid = new_d

    return my_sigmoid

These functions are needed because I would also like to implement my network with this solution, but the three full LUTs do not fit in memory.

In [ ]:
def sigmoid_generator_less_lut(inwidth, outwidth, beta, lut_bits=14):

    my_sigmoid = {}
    beta_val = int(np.exp2(beta))
    lut_range = int(np.exp2(lut_bits))
    shift_bits = inwidth - lut_bits
    
    for i in range(round(-lut_range/2), round(lut_range/2)):
        original_val = i * (2 ** shift_bits)
        my_sigmoid[i] = beta_val / (1 + np.exp(-original_val / beta_val))
    
    plt.figure(figsize=(10, 6))
    plt.plot(list(my_sigmoid.values()))
    plt.title(f"Sigmoid LUT (LUT: {lut_bits} bits, Original: {inwidth} bits, beta={beta})")
    plt.xlabel("Bemenet")
    plt.ylabel("Sigmoid érték")
    plt.grid()
    plt.show()
    
    with open(f"sigmoid_lut_beta{beta}_in{inwidth}_lut{lut_bits}.inc", "w") as f:
        for key in sorted(my_sigmoid.keys()):
            f.write(f"{int(my_sigmoid[key])},\n")
    
    offset = lut_range // 2
    new_d = {k + offset: v for k, v in my_sigmoid.items()}
    my_sigmoid = new_d
    
    return my_sigmoid, shift_bits

In [ ]:
def softplus_generator_less_lut(inwidth, outwidth, beta, lut_bits=14):

    my_softplus = {}
    
    beta_val = int(np.exp2(beta))
    lut_range = int(np.exp2(lut_bits))
    shift_bits = inwidth - lut_bits
    
    function = nn.Softplus(1/beta_val)
    
    for i in range(round(-lut_range/2), round(lut_range/2)):
        original_val = i * (2 ** shift_bits)
        x = torch.tensor(float(original_val))
        my_softplus[i] = function(x).item()
    
    plt.figure(figsize=(10, 6))
    plt.plot(list(my_softplus.values()))
    plt.title(f"Softplus LUT (LUT: {lut_bits} bits, Original: {inwidth} bits, beta={beta})")
    plt.xlabel("Bemenet")
    plt.ylabel("Softplus érték")
    plt.grid()
    plt.show()
    
    with open(f"softplus_lut_beta{beta}_in{inwidth}_lut{lut_bits}.inc", "w") as f:
        for key in sorted(my_softplus.keys()):
            f.write(f"{int(my_softplus[key])},\n")
    
    offset = lut_range // 2
    new_d = {k + offset: v for k, v in my_softplus.items()}
    my_softplus = new_d
    
    return my_softplus, shift_bits

In [ ]:
def tanh_generator_less_lut(inwidth, outwidth, beta, lut_bits=14):

    my_tanh = {}
    
    beta_val = int(np.exp2(beta))
    lut_range = int(np.exp2(lut_bits))
    shift_bits = inwidth - lut_bits
    
    for i in range(round(-lut_range/2), round(lut_range/2)):
        original_val = i * (2 ** shift_bits)
        my_tanh[i] = beta_val * np.tanh(original_val / beta_val)
    
    plt.figure(figsize=(10, 6))
    plt.plot(list(my_tanh.values()))
    plt.title(f"Tanh LUT (LUT: {lut_bits} bits, Original: {inwidth} bits, beta={beta})")
    plt.xlabel("LUT index")
    plt.ylabel("Quantized tanh value")
    plt.grid()
    plt.show()
    
    with open(f"tanh_lut_beta{beta}_in{inwidth}_lut{lut_bits}.inc", "w") as f:
        f.write(f"// Tanh LUT: inwidth={inwidth}, lut_bits={lut_bits}, shift={shift_bits}, beta={beta}\n")
        for key in sorted(my_tanh.keys()):
            f.write(f"{int(my_tanh[key])},\n")
    
    offset = lut_range // 2
    new_d = {k + offset: v for k, v in my_tanh.items()}
    my_tanh = new_d
    
    print(f"Tanh LUT generálva:")
    print(f"  - Eredeti bemenet szélessége: {inwidth} bit")
    print(f"  - LUT címzés: {lut_bits} bit")
    print(f"  - Shiftelés: {shift_bits} bit jobbra")
    print(f"  - LUT méret: {lut_range} elem")
    
    return my_tanh, shift_bits

In [ ]:
inwidth = 18
outwidth = 18
beta = 14
softplus_generator_less_lut(inwidth, outwidth, beta)
inwidth = 19
outwidth = 19
beta = 11
softplus_generator_less_lut(inwidth, outwidth, beta)
inwidth = 19
outwidth = 19
beta = 10
softplus_generator_less_lut(inwidth, outwidth, beta)

In [ ]:
inwidth = 18
outwidth = 18
beta = 14
sigmoid_generator_less_lut(inwidth, outwidth, beta)
inwidth = 19
outwidth = 19
beta = 11
sigmoid_generator_less_lut(inwidth, outwidth, beta)
inwidth = 19
outwidth = 19
beta = 10
sigmoid_generator_less_lut(inwidth, outwidth, beta)

In [ ]:
inwidth = 18
outwidth = 18
beta = 14
tanh_generator_less_lut(inwidth, outwidth, beta)
inwidth = 19
outwidth = 19
beta = 11
tanh_generator_less_lut(inwidth, outwidth, beta)
inwidth = 19
outwidth = 19
beta = 10
tanh_generator_less_lut(inwidth, outwidth, beta)

In [ ]:
inwidth = 12
outwidth = 12
beta = 8
softplus_generator_int(inwidth, outwidth, beta)
inwidth = 15
outwidth = 15
beta = 11
tanh_generator(inwidth, outwidth, beta)
inwidth = 18
outwidth = 18
beta = 14
sigmoid_generator_int(inwidth, outwidth, beta)

#### LUT + Interpolation

First, I define the required helper functions. 

##### Segments required for piecewise linear approximation

In [ ]:
def generate_pwl_switch(name, activation_values, size, num_of_points, shift):

    param = int(size / num_of_points)
    x_values = []
    y_values = []

    for i in range(size):
        if i % param == 0:
            x_values.append(i)
            y_values.append(activation_values[i])

    # last point
    x_values.append(size - 1)
    y_values.append(activation_values[size - 1])

    a = []
    b = []
    k_values = []
    segments = []

    for i in range(len(x_values) - 1):
        ai = (y_values[i+1] - y_values[i]) / (x_values[i+1] - x_values[i])
        bi = y_values[i] - ai * x_values[i]
        k = round(ai * (2**shift))
        a.append(ai)
        b.append(int(round(bi)))
        k_values.append(k)
        segments.append((x_values[i], x_values[i+1], ai, bi, k))

    return segments

##### Segments required for piecewise constant approximation

In [ ]:
def generate_pwc_switch(name, activation_values, size, num_of_points):
    param = int(size / num_of_points)
    x_values = []
    y_values = []

    for i in range(size):
        if i % param == 0:
            x_values.append(i)
            y_values.append(activation_values[i])

    # last point
    x_values.append(size - 1)
    y_values.append(activation_values[size - 1])

    segments = []
    for i in range(len(x_values) - 1):
        yi = y_values[i]
        segments.append((x_values[i], x_values[i+1], yi))
    return segments

This is a more optimal solution, because in the previous case the output of the segment was determined by the first element, while the next solution returns the average of the segment. I will use this in the FPGA implementation.

In [ ]:
def generate_optimal_pwc(activation_values, size, num_of_points):
    param = int(size / num_of_points)
    x_values = []
    y_values = []

    for i in range(size):
        if i % param == 0:
            x_values.append(i)
            y_values.append(activation_values[i])

    x_values.append(size - 1)
    y_values.append(activation_values[size - 1])

    segments = []
    for i in range(len(x_values)-1):
        x0, x1 = x_values[i], x_values[i+1]
        xs = range(x0, x1+1)
        ys = [activation_values[x] for x in xs]
        c = int(round(np.mean(ys)))
        segments.append((x0, x1, c))

    return segments

This function plots the PWL error.

In [ ]:
def plot_pwl_error(activation_values, segments, title="PWL Error"):

    x_all = sorted(activation_values.keys())
    y_orig = np.array([activation_values[x] for x in x_all])

    y_pwl = []
    for x in x_all:
        for (x0, x1, a, b, _) in segments:
            if x0 <= x <= x1:
                y_pwl.append(a * x + b)
                break
    y_pwl = np.array(y_pwl)

    diff = y_orig - y_pwl

    mean_err = np.mean(diff)
    mae = np.mean(np.abs(diff))
    max_err = np.max(np.abs(diff))
    std_err = np.std(diff)

    print("PWL Error:")
    print(f"Átlagos hiba      : {mean_err:.6f}")
    print(f"Átlagos absz. hiba: {mae:.6f}")
    print(f"Maximum hiba      : {max_err:.6f}")
    print(f"Szórás            : {std_err:.6f}")

    plt.figure(figsize=(10,6))
    plt.plot(x_all, diff, color="red", label="Eredeti - PWL")
    plt.axhline(0, color="black", linestyle="--")
    plt.xlabel("Bemenet")
    plt.ylabel("Eltérés")
    plt.title(title)
    plt.grid(True)
    plt.show()

This function plots the PWC error.

In [ ]:
def plot_pwc_error(activation_values, segments, title="PWC Error"):
    x_all = sorted(activation_values.keys())
    y_orig = np.array([activation_values[x] for x in x_all])

    y_pwc = []
    for x in x_all:
        for (x0, x1, yi) in segments:
            if x0 <= x <= x1:
                y_pwc.append(yi)
                break
    y_pwc = np.array(y_pwc)

    diff = y_orig - y_pwc

    mean_err = np.mean(diff)
    mae = np.mean(np.abs(diff))
    max_err = np.max(np.abs(diff))
    std_err = np.std(diff)

    print("PWC Error:")
    print(f"Átlagos hiba      : {mean_err:.6f}")
    print(f"Átlagos absz. hiba: {mae:.6f}")
    print(f"Maximum hiba      : {max_err:.6f}")
    print(f"Szórás            : {std_err:.6f}")

    plt.figure(figsize=(10,6))
    plt.plot(x_all, diff, color="blue", label="Eredeti - PWC")
    plt.axhline(0, color="black", linestyle="--")
    plt.xlabel("Bemenet")
    plt.ylabel("Eltérés")
    plt.title(title)
    plt.grid(True)
    plt.show()

In [ ]:
def draw_pwl(segments):
    plt.figure(figsize=(10,6))

    for (x0, x1, a, b, k) in segments:
        x = np.linspace(x0, x1)
        y = a * x + b
        plt.plot(x, y, label=f"[{x0:.0f},{x1:.0f}]")
        
    plt.xlabel("Bemenet")
    plt.ylabel("Érték")
    plt.title("Szakaszonkénti lineáris közelítés")
    plt.grid(True)
    plt.show()


In [ ]:
def draw_pwc_continuous(segments):
    plt.figure(figsize=(10,6))

    xs = []
    ys = []

    for (x0, x1, c) in segments:
        xs.extend(range(x0, x1+1))
        ys.extend([c] * (x1 - x0 + 1))

    plt.plot(xs, ys, linewidth=2)

    plt.xlabel("Bemenet")
    plt.ylabel("Érték")
    plt.title("PWC közelítés (folytonos rajz)")
    plt.grid(True)
    plt.show()



In [ ]:
def draw_error_pwc(original, segments):
    plt.figure(figsize=(10,6))

    # original curve
    x_orig = sorted(original.keys())
    y_orig = [original[x] for x in x_orig]
    plt.plot(x_orig, y_orig, label="Original", color="black")

    # PWC approximation (single continuous staircase line)
    xs = []
    ys = []

    for (x0, x1, c) in segments:
        xs.extend([x0, x1])
        ys.extend([c, c])

    plt.step(xs, ys, where='post', linewidth=2, label="PWC")

    plt.xlabel("Bemenet")
    plt.ylabel("Érték")
    plt.title("Eredeti és szakaszonkénti konstans közelítés")
    plt.grid(True)
    plt.show()


In [ ]:
def draw_error(original, segments):
    plt.figure(figsize=(10,6))

    x_orig = sorted(original.keys())
    y_orig = [original[x] for x in x_orig]
    plt.plot(x_orig, y_orig, label="Eredeti (original)", color="black")

    for (x0, x1, a, b, k) in segments:
        x = np.linspace(x0, x1, 200)
        y = a * x + b
        plt.plot(x, y, label=f"PWL [{x0:.0f},{x1:.0f}]")

    plt.xlabel("Bemenet")
    plt.ylabel("Érték")
    plt.title("Eredeti és szakaszonkénti lineáris közelítés")
    plt.grid(True)
    plt.show()

In [ ]:
inwidth = 18
outwidth = 18
beta = 14

my_sigmoid = sigmoid_generator(inwidth, outwidth, beta)
segments = generate_pwl_switch("pwl_sigmoid", my_sigmoid, size=int(np.exp2(inwidth)), num_of_points=32, shift=beta)
plot_pwl_error(my_sigmoid, segments, title="Sigmoid PWL közelítés hibája")
draw_pwl(segments)
draw_error(my_sigmoid, segments)

In [ ]:
inwidth = 18
outwidth = 18
beta = 14

my_sigmoid = sigmoid_generator(inwidth, outwidth, beta)
segments = generate_optimal_pwc(my_sigmoid, size=int(np.exp2(inwidth)), num_of_points=32)
plot_pwc_error(my_sigmoid, segments, title="Sigmoid PWL közelítés hibája")
draw_pwc_continuous(segments)
draw_error_pwc(my_sigmoid, segments)

Everything works fine so far, but!

In [ ]:
inwidth = 19
outwidth = 19
beta = 11

my_tanh = tanh_generator(inwidth, outwidth, beta)
segments = generate_pwl_switch("pwl_tanh", my_tanh, size=int(np.exp2(inwidth)), num_of_points=32, shift=beta)
plot_pwl_error(my_tanh, segments, title="Tanh PWL közelítés hibája")
draw_pwl(segments)
draw_error(my_tanh, segments)

The approach above works well when the difference between the input bit width and the scaling factor is 4 or less. If it is greater than 4, then near the edges of the function many intervals take either 0 or the maximum value, and in the interval where change occurs the approximation becomes coarser, so the error is higher. Another issue is that a separate function must be defined for every bit width and scaling factor.
Instead, another approach: define an activation function with a large bit width but ideal scaling factor and bit depth, and compute lower-sf activation functions by scaling this one. Since for all three problematic activation functions the 'interesting' part is in the middle, it is enough to examine that interval; so if it shifts outside the examined interval in the negative direction we assign the minimum value, and in the positive direction we assign the maximum value (which becomes the identity function for softplus). For scaling, I use the following formula: 
$y = \\left(\\frac{x \\cdot k}{2^{\\Delta}} \\right) \\gg (\\text{defaultSf} - \\Delta) + b \\cdot 2^{\\Delta}$, where $k = a \\cdot 2^{defaultSf}$, defaultSf denotes the scaling factor of the original activation, and $\\Delta$ is the deviation from it.

In [ ]:
def generate_pwl_switch_python(name, activation_values, size, num_of_points, shift):

    param = int(size / num_of_points)
    x_values = []
    y_values = []

    for i in range(size):
        if i % param == 0:
            x_values.append(i)
            y_values.append(activation_values[i])

    # last point
    x_values.append(size - 1)
    y_values.append(activation_values[size - 1])

    a = []
    b = []
    k_values = []
    segments = []

    for i in range(len(x_values) - 1):
        ai = (y_values[i+1] - y_values[i]) / (x_values[i+1] - x_values[i])
        bi = y_values[i] - ai * x_values[i]
        k = round(ai * (2**shift))
        a.append(ai)
        b.append(int(round(bi)))
        k_values.append(k)
        segments.append((x_values[i], x_values[i+1], ai, bi, k))

    print(f"def {name}(x, scale):")
    print(f"    delta = {shift} - scale")
    print(f"    interval = x >> {shift} - 1 - delta")
    print(f"    match interval:")

    for idx, (k, bi) in enumerate(zip(k_values, b)):
        print(f"        case {idx}: return ((x * {k}) >> {shift}) + ({bi} >> delta)")
    print(f"        case _: {size} << delta")

    return segments

In [ ]:
inwidth = 19
outwidth = 19
beta = 15

my_sigmoid = sigmoid_generator(inwidth, outwidth, beta)
segments = generate_pwl_switch_python("pwl_sigmoid", my_sigmoid, size=int(np.exp2(inwidth)), num_of_points=32, shift=beta)

In [ ]:
def pwl_sigmoid(x, scale):
    delta = 15 - scale
    interval = x >> 15 - 1 - delta
    match interval:
        case 0: return ((x * 14) >> 15) + (11 >> delta)
        case 1: return ((x * 23) >> 15) + (6 >> delta)
        case 2: return ((x * 39) >> 15) + (-9 >> delta)
        case 3: return ((x * 64) >> 15) + (-46 >> delta)
        case 4: return ((x * 105) >> 15) + (-128 >> delta)
        case 5: return ((x * 172) >> 15) + (-296 >> delta)
        case 6: return ((x * 281) >> 15) + (-625 >> delta)
        case 7: return ((x * 459) >> 15) + (-1245 >> delta)
        case 8: return ((x * 742) >> 15) + (-2380 >> delta)
        case 9: return ((x * 1187) >> 15) + (-4381 >> delta)
        case 10: return ((x * 1863) >> 15) + (-7763 >> delta)
        case 11: return ((x * 2841) >> 15) + (-13138 >> delta)
        case 12: return ((x * 4143) >> 15) + (-20954 >> delta)
        case 13: return ((x * 5670) >> 15) + (-30877 >> delta)
        case 14: return ((x * 7117) >> 15) + (-41007 >> delta)
        case 15: return ((x * 8025) >> 15) + (-47820 >> delta)
        case 16: return ((x * 8025) >> 15) + (-47820 >> delta)
        case 17: return ((x * 7117) >> 15) + (-40099 >> delta)
        case 18: return ((x * 5670) >> 15) + (-27074 >> delta)
        case 19: return ((x * 4143) >> 15) + (-12572 >> delta)
        case 20: return ((x * 2841) >> 15) + (456 >> delta)
        case 21: return ((x * 1863) >> 15) + (10717 >> delta)
        case 22: return ((x * 1187) >> 15) + (18156 >> delta)
        case 23: return ((x * 742) >> 15) + (23271 >> delta)
        case 24: return ((x * 459) >> 15) + (26674 >> delta)
        case 25: return ((x * 281) >> 15) + (28890 >> delta)
        case 26: return ((x * 172) >> 15) + (30314 >> delta)
        case 27: return ((x * 105) >> 15) + (31221 >> delta)
        case 28: return ((x * 64) >> 15) + (31796 >> delta)
        case 29: return ((x * 39) >> 15) + (32158 >> delta)
        case 30: return ((x * 23) >> 15) + (32386 >> delta)
        case 31: return ((x * 14) >> 15) + (32529 >> delta)
        case _: 524288 << delta

In [ ]:
diff = -7
inwidth += diff
outwidth += diff
beta += diff

print(beta)
pwl_sigmoid_dict = {}
for i in range(0, int(np.exp2(inwidth))):
    pwl_sigmoid_dict[i] = pwl_sigmoid(i, beta)

plt.plot(list(pwl_sigmoid_dict.values()))
plt.grid()
plt.show()

my_sigmoid = sigmoid_generator(inwidth, outwidth, beta)

pwl_sigmoid_error = {}
for i in range(0, int(np.exp2(inwidth))):
    pwl_sigmoid_error[i] = pwl_sigmoid_dict[i] - my_sigmoid[i]

plt.plot(list(pwl_sigmoid_error.values()))
plt.grid()
plt.show()


From the run above, it can be seen that this solution can work well even with a low sf. The following function creates an array that can be copied directly into HLS.

In [ ]:
def generate_pwl_switch_hls(name, activation_values, size, num_of_points, shift):

    param = int(size / num_of_points)
    x_values = []
    y_values = []

    for i in range(size):
        if i % param == 0:
            x_values.append(i)
            y_values.append(activation_values[i])

    x_values.append(size - 1)
    y_values.append(activation_values[size - 1])

    a = []
    b = []
    k_values = []
    segments = []

    for i in range(len(x_values) - 1):
        ai = (y_values[i+1] - y_values[i]) / (x_values[i+1] - x_values[i])
        bi = y_values[i] - ai * x_values[i]
        k = round(ai * (2**shift))
        a.append(ai)
        b.append(int(round(bi)))
        k_values.append(k)
        segments.append((x_values[i], x_values[i+1], ai, bi, k))

    print(f"static const int {name}[32][2] = {{")

    for idx, (k, bi) in enumerate(zip(k_values, b)):
        print(f"    {{{k}, {bi}}},")

    print("};")    

    return segments

In [ ]:
def generate_pwc_switch_hls(name, activation_values, size, num_of_points):
    param = int(size / num_of_points)
    x_values = []
    y_values = []

    for i in range(size):
        if i % param == 0:
            x_values.append(i)
            y_values.append(activation_values[i])

    x_values.append(size - 1)
    y_values.append(activation_values[size - 1])

    segments = []
    constants = []

    for i in range(len(x_values)-1):
        x0, x1 = x_values[i], x_values[i+1]
        xs = range(x0, x1+1)
        ys = [activation_values[x] for x in xs]
        c = int(round(np.mean(ys)))
        segments.append((x0, x1, c))
        constants.append(c)

    print(f"static const int {name}[32] = {{")
    for c in constants:
        print(f"    {c},")
    print("};")

    return segments

In [ ]:
def segment_writer(name, segments):
    with open(f"segments_4_{name}.txt", "w") as f:
        for segment in segments:
            f.write(f"{segment[4]}, {round(segment[3])}\n")

The following code snippets create the arrays used in HLS.

In [ ]:
inwidth = 18
outwidth = 18
beta = 14

my_softplus = softplus_generator(inwidth, outwidth, beta)
segments = generate_pwl_switch_hls("pwl_softplus", my_softplus, size=int(np.exp2(inwidth)), num_of_points=32, shift=beta)
segment_writer("my_softplus_pwl", segments)
segments = generate_pwc_switch_hls("pwl_softplus", my_softplus, size=int(np.exp2(inwidth)), num_of_points=32)

In [ ]:
inwidth = 18
outwidth = 18
beta = 14

my_sigmoid = sigmoid_generator(inwidth, outwidth, beta)
segments = generate_pwl_switch_hls("pwl_sigmoid", my_sigmoid, size=int(np.exp2(inwidth)), num_of_points=32, shift=beta)
segment_writer("my_sigmoid", segments)
segments = generate_pwc_switch_hls("pwl_sigmoid", my_sigmoid, size=int(np.exp2(inwidth)), num_of_points=32)

In [ ]:
inwidth = 18
outwidth = 18
beta = 14

my_tanh = tanh_generator(inwidth, outwidth, beta)
segments = generate_pwl_switch_hls("pwl_tanh", my_tanh, size=int(np.exp2(inwidth)), num_of_points=32, shift=beta)
segment_writer("my_tanh", segments)
segments = generate_pwc_switch_hls("pwl_tanh", my_tanh, size=int(np.exp2(inwidth)), num_of_points=32)
